# Sentiment Analysis with OpenRouter

This notebook runs sentiment analysis on entity files using **OpenRouter**, which provides access to models from:
- Meta (Llama 3.x)
- Mistral (Mistral, Mixtral)
- OpenAI (GPT-4o, GPT-3.5)
- Anthropic (Claude 3.x)
- Google (Gemini)
- Qwen, DeepSeek, and more

## Setup
1. Get an API key from [OpenRouter](https://openrouter.ai/keys)
2. Set the `OPENROUTER_API_KEY` environment variable or pass it directly

In [ ]:
# Install dependencies if needed
# !pip install openai pandas

## 1. Setup and Configuration

In [ ]:
import os
import pandas as pd
from pathlib import Path
from openrouter_adapter import OpenRouterClient, AVAILABLE_MODELS, normalize_prediction

# Set your API key (or use environment variable OPENROUTER_API_KEY)
API_KEY = os.environ.get("OPENROUTER_API_KEY", "your-api-key-here")

# Paths
DATA_DIR = Path("data")
INPUT_DIR = DATA_DIR / "per_entity"
OUTPUT_BASE_DIR = DATA_DIR / "per_entity_llm"

print(f"Input directory: {INPUT_DIR}")
print(f"Output base directory: {OUTPUT_BASE_DIR}")

Input directory: data/per_entity
Output base directory: data/per_entity_llm


## 2. Choose Models to Run

Select which models you want to use. Each model will create its own output folder.

In [3]:
# Print all available models
print("Available models (shorthand -> full ID):")
print("=" * 60)
for short, full in sorted(AVAILABLE_MODELS.items()):
    print(f"  {short:20} -> {full}")

Available models (shorthand -> full ID):
  claude-3-haiku       -> anthropic/claude-3-haiku
  claude-3-opus        -> anthropic/claude-3-opus
  claude-3.5-sonnet    -> anthropic/claude-3.5-sonnet
  command-r            -> cohere/command-r
  command-r-plus       -> cohere/command-r-plus
  deepseek-r1          -> deepseek/deepseek-r1
  deepseek-v3          -> deepseek/deepseek-chat
  gemini-1.5-flash     -> google/gemini-flash-1.5
  gemini-1.5-pro       -> google/gemini-pro-1.5
  gemini-2.0-flash     -> google/gemini-2.0-flash-001
  gemma-2-27b          -> google/gemma-2-27b-it
  gemma-2-9b           -> google/gemma-2-9b-it
  gpt-3.5-turbo        -> openai/gpt-3.5-turbo
  gpt-4-turbo          -> openai/gpt-4-turbo
  gpt-4o               -> openai/gpt-4o
  gpt-4o-mini          -> openai/gpt-4o-mini
  llama-3-70b          -> meta-llama/llama-3-70b-instruct
  llama-3-8b           -> meta-llama/llama-3-8b-instruct
  llama-3.1-405b       -> meta-llama/llama-3.1-405b-instruct
  llama-3.1-70b  

In [4]:
# Select models to run (use shorthands or full model IDs)
MODELS_TO_RUN = [
    "llama-3.1-8b",       # Fast and cheap
    # "mistral-7b",         # Good balance
    # "gpt-4o-mini",      # OpenAI option
    # "claude-3-haiku",   # Anthropic option
    # "gemini-1.5-flash", # Google option
    # "qwen-2.5-7b",      # Qwen option
]

print(f"Will run {len(MODELS_TO_RUN)} model(s): {MODELS_TO_RUN}")

Will run 1 model(s): ['llama-3.1-8b']


## 3. Initialize OpenRouter Client

In [8]:
# Initialize the client
client = OpenRouterClient(
    api_key=API_KEY,
    site_name="TEMPO-BIAS",  # Optional: shows up in OpenRouter analytics
)

# Test the connection
test_response = client.generate(
    prompt="Say 'hello' and nothing else.",
    model="llama-3.1-8b",
    max_tokens=10,
)
print(f"Connection test: {test_response}")

Connection test: Hello


## 4. Define Sentiment Analysis Function

In [9]:
def run_sentiment_prompt(client: OpenRouterClient, prompt_text: str, model: str) -> str:
    """
    Run sentiment analysis on a single prompt.
    
    Args:
        client: OpenRouterClient instance
        prompt_text: The full sentiment analysis prompt
        model: Model shorthand or full ID
    
    Returns:
        Normalized sentiment prediction
    """
    try:
        response = client.generate(
            prompt=prompt_text,
            model=model,
            temperature=0,  # Deterministic
            max_tokens=16,  # Just need one word
        )
        return normalize_prediction(response)
    except Exception as e:
        print(f"Error: {e}")
        return "error"


# Test on one example
test_prompt = """Analyze the sentiment towards the target of the following sentence and classify it into one of the following categories:

negative for Negative sentiment

neutral for Neutral sentiment

positive for Positive sentiment

Please provide only the sentiment score based on the provided scale. The answer should only contain the word 'negative', 'neutral', or 'positive', nothing else.

Sentence: Angela Merkel was credited with helping ease tensions in ongoing negotiations.

Target: Angela Merkel

Sentiment:"""

result = run_sentiment_prompt(client, test_prompt, "llama-3.1-8b")
print(f"Test sentiment: {result} (expected: positive)")

Test sentiment: positive (expected: positive)


## 5. Process All Entities with Selected Models

In [10]:
def get_model_folder_name(model: str) -> str:
    """Convert model name to a safe folder name."""
    # Use shorthand if available, otherwise sanitize the full ID
    name = model.replace("/", "_").replace(".", "-")
    return name


def process_entities_for_model(
    client: OpenRouterClient,
    model: str,
    input_dir: Path,
    output_dir: Path,
    max_entities: int = None,  # Set to limit for testing
):
    """
    Process all entity files with a specific model.
    
    Args:
        client: OpenRouterClient instance
        model: Model to use
        input_dir: Directory with entity CSVs
        output_dir: Directory to save results
        max_entities: Limit number of entities (for testing)
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    entity_files = sorted(input_dir.glob("*.csv"))
    
    if max_entities:
        entity_files = entity_files[:max_entities]
    
    print(f"\nProcessing {len(entity_files)} entities with model: {model}")
    print(f"Output directory: {output_dir}")
    print("-" * 60)
    
    for i, csv_path in enumerate(entity_files):
        df = pd.read_csv(csv_path)
        predictions = []
        
        for prompt_text in df["prompt"]:
            pred = run_sentiment_prompt(client, prompt_text, model)
            predictions.append(pred)
        
        df["predicted_sentiment"] = predictions
        df["model"] = model  # Track which model made predictions
        
        out_path = output_dir / csv_path.name
        df.to_csv(out_path, index=False)
        
        if (i + 1) % 10 == 0 or i == 0 or i == len(entity_files) - 1:
            print(f"  [{i + 1}/{len(entity_files)}] {csv_path.name}")
    
    print(f"Done! Saved {len(entity_files)} files to {output_dir}")

In [11]:
# Run for all selected models
# Set max_entities to a small number (e.g., 5) for testing
MAX_ENTITIES = 1  # Set to None to process all entities

for model in MODELS_TO_RUN:
    model_folder = get_model_folder_name(model)
    output_dir = OUTPUT_BASE_DIR / model_folder
    
    process_entities_for_model(
        client=client,
        model=model,
        input_dir=INPUT_DIR,
        output_dir=output_dir,
        max_entities=MAX_ENTITIES,
    )


Processing 1 entities with model: llama-3.1-8b
Output directory: data/per_entity_llm/llama-3-1-8b
------------------------------------------------------------
  [1/1] Abdel_Fattah_el-Sisi.csv
Done! Saved 1 files to data/per_entity_llm/llama-3-1-8b


## 6. Compare Results Across Models

In [ ]:
def compare_model_results(output_base_dir: Path, entity_name: str):
    """
    Compare predictions from different models for a single entity.
    """
    filename = entity_name.replace(" ", "_") + ".csv"
    results = []
    
    for model_dir in output_base_dir.iterdir():
        if model_dir.is_dir():
            file_path = model_dir / filename
            if file_path.exists():
                df = pd.read_csv(file_path)
                model_name = model_dir.name
                
                # Calculate accuracy
                correct = (df["expected_sentiment"] == df["predicted_sentiment"]).sum()
                total = len(df)
                accuracy = correct / total * 100
                
                results.append({
                    "model": model_name,
                    "accuracy": accuracy,
                    "correct": correct,
                    "total": total,
                })
    
    if results:
        results_df = pd.DataFrame(results)
        results_df = results_df.sort_values("accuracy", ascending=False)
        print(f"Results for: {entity_name}")
        print(results_df.to_string(index=False))
        return results_df
    else:
        print(f"No results found for {entity_name}")
        return None


# Example comparison
compare_model_results(OUTPUT_BASE_DIR, "Angela_Merkel")

In [ ]:
def aggregate_model_accuracy(output_base_dir: Path):
    """
    Calculate overall accuracy for each model across all entities.
    """
    model_stats = {}
    
    for model_dir in output_base_dir.iterdir():
        if model_dir.is_dir():
            model_name = model_dir.name
            total_correct = 0
            total_predictions = 0
            
            for csv_file in model_dir.glob("*.csv"):
                df = pd.read_csv(csv_file)
                if "predicted_sentiment" in df.columns and "expected_sentiment" in df.columns:
                    correct = (df["expected_sentiment"] == df["predicted_sentiment"]).sum()
                    total_correct += correct
                    total_predictions += len(df)
            
            if total_predictions > 0:
                model_stats[model_name] = {
                    "accuracy": total_correct / total_predictions * 100,
                    "correct": total_correct,
                    "total": total_predictions,
                }
    
    if model_stats:
        stats_df = pd.DataFrame(model_stats).T
        stats_df = stats_df.sort_values("accuracy", ascending=False)
        print("Overall Model Accuracy:")
        print("=" * 60)
        print(stats_df.to_string())
        return stats_df
    return None


aggregate_model_accuracy(OUTPUT_BASE_DIR)

## 7. Quick Usage Examples

Here are some examples of using the OpenRouter client directly:

In [ ]:
# Example 1: Simple generation
response = client.generate(
    prompt="Explain quantum computing in one sentence.",
    model="llama-3.1-8b",
    max_tokens=50,
)
print(f"Llama 3.1 8B: {response}")

In [ ]:
# Example 2: Compare same prompt across models
prompt = "What is 2 + 2? Answer with just the number."

for model in ["llama-3.1-8b", "mistral-7b"]:
    response = client.generate(prompt=prompt, model=model, max_tokens=10)
    print(f"{model}: {response}")

In [ ]:
# Example 3: Use full model ID directly
response = client.generate(
    prompt="Hello!",
    model="meta-llama/llama-3.1-70b-instruct",  # Full OpenRouter model ID
    max_tokens=30,
)
print(f"Response: {response}")